This is a notebook to visualize different properties of the dataset as we go about curating. Here, we're using datasets that are often used in the Bio-ML community, provided in this case by [biomap research](https://huggingface.co/biomap-research) as part of XTrimo-PGLM. These are biology benchmarks, so they're generally pretty bad.

In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import polars as pl
from project.utils.strs import SEED, data_dir

In [ ]:
subset_dir = data_dir / 'processed_subsets'
bio_dir = subset_dir / 'biomap-research'
bio_dir.mkdir(parents=True, exist_ok=True)

We want to read each dataset and make a validation set if necessary,by randomly splitting some data off of the training set

In [ ]:
def preprocess_biomap_dataset(
                            dataset_name: str,
                            val_size: float = 0.1,
                            seed: int = 18252,
):
    """ 
    Dataset names that have been tried are:
    ssp_q3
    ssp_q8
    fold_prediction
    fitness_prediction
    metal_ion_binding
    localization_prediction
    enzyme_catalytic_efficiency
    optimal_ph
    optimal_temperature
    temperature_stability
    stability_prediction
    solubility_prediction
    """
    import polars as pl
    import uuid

    biomap_download_path = f'hf://datasets/biomap-research/{dataset_name}/'

    splits = {
        'train': 'data/train-00000-of-00001.parquet', 
        'valid': 'data/valid-00000-of-00001.parquet', 
        'test': 'data/test-00000-of-00001.parquet'}

    # Read the train dataset
    train_df = pl.read_parquet(biomap_download_path + splits['train'])

    # Read the val dataset, if available
    try:
        val_df = pl.read_parquet(biomap_download_path + splits['valid'])
    except:
        print('unable to get validation set! Randomly splitting the train set')
        # Randomly train:test split from train_df
        from sklearn.model_selection import train_test_split
        # Convert to pandas for sklearn compatibility
        df_pandas = train_df.to_pandas()
        # Randomly split some data off from the train set
        train_df, val_df = train_test_split(df_pandas, test_size=val_size, random_state=seed)
        # Convert back to polars
        train_df = pl.from_pandas(train_df)
        val_df = pl.from_pandas(val_df)

    # Read the test dataset
    test_df = pl.read_parquet(biomap_download_path + splits['test'])

    # Add the split column
    datasets = {'train': train_df, 'val': val_df, 'test': test_df}
    for k,v in datasets.items():
        datasets[k] = v.with_columns(
            split = pl.lit(k)
        )
    # Combine datasets
    combined_datasets = pl.concat([v for v in datasets.values()], how='vertical').rename({'label': 'targets', 'seq': 'sequence'})

    # Create a uuid for each sample so we have an 'id' column
    combined_datasets = combined_datasets.with_columns(
        pl.Series([str(uuid.uuid4()) for _ in range(len(combined_datasets))]).alias("id")
    )

    # Put things in the order we'll expect later, and return
    return combined_datasets.select(["id", "sequence", "targets", "split"])

### Subsetting

In [ ]:
for dataset in ['ssp_q3',
                'ssp_q8',
                # 'fold_prediction',
                # 'fitness_prediction',
                'metal_ion_binding',
                'localization_prediction',
                # 'enzyme_catalytic_efficiency',
                # 'optimal_ph',
                # 'optimal_temperature',
                # 'temperature_stability',
                # 'stability_prediction',
                # 'solubility_prediction',
                ]:
    print(dataset)
    save_name = bio_dir / f"{dataset}_processed.parquet.gz"
    if not save_name.exists():
        df = preprocess_biomap_dataset(dataset, seed=SEED)
        df.write_parquet(save_name)
    else:
        df = pl.read_parquet(save_name)
    print(df['split'].value_counts(normalize=True))
    # Make a subset with only sequences less than or equal to 512, for AMPLIFY checkpointing experiments
    df_512 = df.filter(pl.col('sequence').str.len_chars() <= 512)
    print('after subsetting')
    print(df_512['split'].value_counts(normalize=True))

    df_512.write_parquet(bio_dir / f"{dataset}_processed_512_cutoff.parquet.gz")